# Notebook 02 — CNN Classifier Training (Subject Model)
### XAI for Music Genre Classification | GTZAN Dataset
---
**Purpose:** Train a CNN on mel spectrograms. This is the *subject model* we will explain — not the paper's contribution itself.

**Outputs:**
- `../models/cnn_gtzan.pt` — trained PyTorch model
- `../models/label_encoder.pkl` — genre → int mapping
- `../figures/02_*.png` — training curves, confusion matrix, per-class metrics (300 DPI)

**Architecture:** Small CNN (3 conv blocks + 2 FC layers). Designed to be transparent and replicable rather than accuracy-maximising — this is intentional for XAI research.

In [ ]:
# ── Cell 1: Installs ──────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['torch', 'torchvision', 'librosa', 'numpy', 'pandas',
            'matplotlib', 'seaborn', 'scikit-learn', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Done.')

In [ ]:
# ── Cell 2: Imports & config ──────────────────────────────────────────────────
import os, pickle, warnings
import numpy as np
import pandas as pd
import librosa
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# Publication plot config
plt.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300,
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 11, 'figure.titlesize': 16,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

FIGURES = Path('../figures')
MODELS  = Path('../models')
DATA    = Path('../data')
FIGURES.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

DEVICE = (
    'mps' if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print(f'Device: {DEVICE}')

GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']

GENRE_COLORS = dict(zip(GENRES, [
    '#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
    '#00BCD4','#E91E63','#8BC34A','#FF5722','#607D8B'
]))

# Audio / spectrogram config
SR       = 22050
DURATION = 30.0
N_MELS   = 128
HOP      = 512
N_FFT    = 2048

def savefig(name, fig=None):
    if fig is None: fig = plt.gcf()
    path = FIGURES / f'{name}.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'  ✓ Saved {path}')
    plt.show()

In [ ]:
# ── Cell 3: Dataset class ─────────────────────────────────────────────────────
class GTZANDataset(Dataset):
    """
    Loads GTZAN clips as fixed-size mel spectrogram tensors.
    Supports 3-second segment sampling for data augmentation.
    """
    def __init__(self, meta_df, label_encoder, segment_s=3.0, augment=False):
        self.records  = meta_df.reset_index(drop=True)
        self.le       = label_encoder
        self.seg_len  = int(segment_s * SR)
        self.augment  = augment
        self.fixed_w  = 128  # fixed time width for CNN

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row   = self.records.iloc[idx]
        label = self.le.transform([row.genre])[0]

        try:
            y, _ = librosa.load(row.filepath, sr=SR, duration=DURATION)
        except Exception:
            y = np.zeros(int(SR * DURATION))

        # Random segment for augmentation
        if self.augment and len(y) > self.seg_len:
            start = np.random.randint(0, len(y) - self.seg_len)
            y = y[start:start + self.seg_len]
        else:
            y = y[:self.seg_len] if len(y) >= self.seg_len else np.pad(y, (0, self.seg_len - len(y)))

        # Mel spectrogram
        S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP)
        S_db = librosa.power_to_db(S, ref=np.max)

        # Normalise to [0, 1]
        S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8)

        # Resize time axis to fixed width
        if S_db.shape[1] != self.fixed_w:
            from skimage.transform import resize
            S_db = resize(S_db, (N_MELS, self.fixed_w), anti_aliasing=True)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)  # (1, 128, 128)
        return tensor, label, row.filename

print('GTZANDataset defined.')

In [ ]:
# ── Cell 4: CNN Architecture ──────────────────────────────────────────────────
class MusicCNN(nn.Module):
    """
    3-block CNN for mel spectrogram classification.
    Designed for interpretability: no residual connections,
    explicit spatial feature maps accessible for XAI.
    """
    def __init__(self, n_classes=10, dropout=0.4):
        super().__init__()

        def conv_block(in_ch, out_ch, pool=(2,2)):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(pool),
                nn.Dropout2d(p=0.15),
            )

        self.block1 = conv_block(1, 32)
        self.block2 = conv_block(32, 64)
        self.block3 = conv_block(64, 128)

        self.pool   = nn.AdaptiveAvgPool2d((4, 4))
        self.flat   = nn.Flatten()
        self.head   = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        x = self.flat(x)
        return self.head(x)

    def get_activations(self, x):
        """Return intermediate feature maps for XAI."""
        acts = {}
        x = self.block1(x);  acts['block1'] = x.detach()
        x = self.block2(x);  acts['block2'] = x.detach()
        x = self.block3(x);  acts['block3'] = x.detach()
        return acts

model = MusicCNN(n_classes=10).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'MusicCNN: {total_params:,} parameters')
print(model)

In [ ]:
# ── Cell 5: Build dataloaders ─────────────────────────────────────────────────
# Re-load metadata from notebook 01
meta = pd.read_csv(DATA / 'gtzan_metadata.csv')
clean_meta = meta[meta.fault == 'clean'].copy()

le = LabelEncoder()
le.fit(GENRES)
with open(MODELS / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(clean_meta, clean_meta['genre']):
    train_meta = clean_meta.iloc[train_idx]
    test_meta  = clean_meta.iloc[test_idx]

BATCH = 32

train_ds = GTZANDataset(train_meta, le, segment_s=3.0, augment=True)
test_ds  = GTZANDataset(test_meta,  le, segment_s=3.0, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ── Cell 6: Training loop ─────────────────────────────────────────────────────
EPOCHS   = 60
LR       = 1e-3
WD       = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for X, y, _ in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * len(y)
        t_correct += (logits.argmax(1) == y).sum().item()
        t_total   += len(y)

    # ── Validate ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for X, y, _ in test_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            v_loss    += loss.item() * len(y)
            v_correct += (logits.argmax(1) == y).sum().item()
            v_total   += len(y)

    scheduler.step()

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    history['train_loss'].append(t_loss / t_total)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss / v_total)
    history['val_acc'].append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), MODELS / 'cnn_gtzan_best.pt')

    if epoch % 10 == 0 or epoch == 1:
        print(f'Ep {epoch:03d} | train_loss={t_loss/t_total:.4f} acc={t_acc:.3f} | '
              f'val_loss={v_loss/v_total:.4f} acc={v_acc:.3f}')

torch.save(model.state_dict(), MODELS / 'cnn_gtzan.pt')
print(f'\nBest val accuracy: {best_val_acc:.4f}')

In [ ]:
# ── Cell 7: Figure 08 — Training curves ──────────────────────────────────────
epochs_range = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(epochs_range, history['train_loss'], label='Train loss', color='#2196F3', linewidth=2)
ax1.plot(epochs_range, history['val_loss'],   label='Val loss',   color='#F44336', linewidth=2, linestyle='--')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('A  Loss Curves')
ax1.legend()

ax2.plot(epochs_range, [a*100 for a in history['train_acc']], label='Train acc', color='#2196F3', linewidth=2)
ax2.plot(epochs_range, [a*100 for a in history['val_acc']],   label='Val acc',   color='#F44336', linewidth=2, linestyle='--')
ax2.axhline(best_val_acc*100, color='gray', linestyle=':', linewidth=1.2, label=f'Best val {best_val_acc*100:.1f}%')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('B  Accuracy Curves')
ax2.legend()

fig.suptitle('Figure 8 — CNN Training History on GTZAN (fault-clean split)', fontweight='bold')
savefig('02_training_curves', fig)

In [ ]:
# ── Cell 8: Full evaluation on test set ──────────────────────────────────────
model.load_state_dict(torch.load(MODELS / 'cnn_gtzan_best.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_files = [], [], []
with torch.no_grad():
    for X, y, fnames in test_loader:
        X = X.to(DEVICE)
        logits = model(X)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())
        all_files.extend(fnames)

pred_genres  = le.inverse_transform(all_preds)
true_genres  = le.inverse_transform(all_labels)

report = classification_report(true_genres, pred_genres, target_names=GENRES, output_dict=True)
print(classification_report(true_genres, pred_genres, target_names=GENRES))

In [ ]:
# ── Cell 9: Figure 09 — Confusion matrix ─────────────────────────────────────
cm = confusion_matrix(true_genres, pred_genres, labels=GENRES)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['A  Raw Counts', 'B  Row-Normalised (Recall)']):
    sns.heatmap(
        data, ax=ax, annot=True, fmt=fmt,
        xticklabels=[g.capitalize() for g in GENRES],
        yticklabels=[g.capitalize() for g in GENRES],
        cmap='Blues', linewidths=0.5,
        cbar_kws={'shrink': 0.75}
    )
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('True', fontweight='bold')
    ax.set_title(title, fontweight='bold')

fig.suptitle('Figure 9 — Confusion Matrix: MusicCNN on GTZAN Test Set', fontweight='bold')
savefig('02_confusion_matrix', fig)

In [ ]:
# ── Cell 10: Figure 10 — Per-class metrics bar chart ─────────────────────────
metrics_df = pd.DataFrame({
    'Genre':     GENRES,
    'Precision': [report[g]['precision'] for g in GENRES],
    'Recall':    [report[g]['recall']    for g in GENRES],
    'F1-Score':  [report[g]['f1-score']  for g in GENRES],
})

fig, ax = plt.subplots(figsize=(16, 7))
x   = np.arange(len(GENRES))
w   = 0.25
colors = ['#2196F3', '#4CAF50', '#F44336']
metrics = ['Precision', 'Recall', 'F1-Score']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*w, metrics_df[metric], w, label=metric, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + w)
ax.set_xticklabels([g.capitalize() for g in GENRES], rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)
ax.set_title('Figure 10 — Per-Class Classification Metrics', fontweight='bold')
ax.legend()

# Annotate F1 on top
for i, (genre, f1) in enumerate(zip(GENRES, metrics_df['F1-Score'])):
    ax.text(i + 2*w, f1 + 0.02, f'{f1:.2f}', ha='center', fontsize=8, color='#F44336', fontweight='bold')

savefig('02_per_class_metrics', fig)

In [ ]:
# ── Cell 11: Save predictions for XAI notebooks ──────────────────────────────
pred_df = pd.DataFrame({
    'filename':   all_files,
    'true_genre': true_genres,
    'pred_genre': pred_genres,
    'correct':    (np.array(true_genres) == np.array(pred_genres)).astype(int)
})
pred_df = pred_df.merge(
    pd.read_csv(DATA / 'gtzan_metadata.csv')[['filename', 'filepath', 'fault']],
    on='filename', how='left'
)
pred_df.to_csv(DATA / 'test_predictions.csv', index=False)

overall_acc = pred_df['correct'].mean()
print(f'Overall test accuracy: {overall_acc:.4f}')
print(f'Predictions saved to ../data/test_predictions.csv')
print(f'Model saved to ../models/cnn_gtzan_best.pt')